# Bitcoin Trading Strategy - Deep Neural Network Improvement

## 1. Environment Setup & Data Loading
We use the updated `yfinance` library to load data and `utils.py` for preprocessing. We also use `optuna` for hyperparameter optimization.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import optuna
from optuna.visualization import plot_optimization_history, plot_param_importances

from utils import (
    load_bitcoin_data,
    create_features,
    prepare_data,
    evaluate_model,
    plot_confusion_matrix,
    device
)

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)

# Load Data
start_date = "2020-01-01"
end_date = datetime.now().strftime("%Y-%m-%d")

print(f"Loading data from {start_date} to {end_date}...")
btc_data = load_bitcoin_data(start_date=start_date, end_date=end_date)
btc_features = create_features(btc_data, lookback_days=10)

print(f"Data Shape: {btc_features.shape}")

## 2. Data Preparation
Scaling and sequence creation for the RNN model.

In [ ]:
# Split and Scale
X_train, X_val, X_test, y_train, y_val, y_test = prepare_data(
    btc_features, test_size=0.2, validation_size=0.1
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# Create Sequences
sequence_length = 30

def create_sequences(X, y, seq_len=30):
    X_seq, y_seq = [], []
    for i in range(len(X) - seq_len):
        X_seq.append(X[i:i+seq_len])
        y_seq.append(y[i+seq_len])
    return np.array(X_seq), np.array(y_seq)

X_train_seq, y_train_seq = create_sequences(X_train_scaled, y_train.values, sequence_length)
X_val_seq, y_val_seq = create_sequences(X_val_scaled, y_val.values, sequence_length)
X_test_seq, y_test_seq = create_sequences(X_test_scaled, y_test.values, sequence_length)

# DataLoaders
train_loader = DataLoader(TensorDataset(torch.FloatTensor(X_train_seq), torch.FloatTensor(y_train_seq)), batch_size=32, shuffle=True)
val_loader = DataLoader(TensorDataset(torch.FloatTensor(X_val_seq), torch.FloatTensor(y_val_seq)), batch_size=32, shuffle=False)
test_loader = DataLoader(TensorDataset(torch.FloatTensor(X_test_seq), torch.FloatTensor(y_test_seq)), batch_size=32, shuffle=False)

## 3. Improved GRU Model & Optuna Optimization
We use Optuna to find the best hyperparameters (hidden size, dropout, learning rate, and **threshold**).
**Objective**: Maximize Total Return (Profit) on Validation Set.

In [ ]:
class ImprovedGRUModel(nn.Module):
    def __init__(self, input_size, hidden_size=64, dropout=0.2):
        super(ImprovedGRUModel, self).__init__()
        
        self.gru1 = nn.GRU(input_size, hidden_size, batch_first=True, num_layers=1)
        self.dropout1 = nn.Dropout(dropout)
        self.bn1 = nn.BatchNorm1d(hidden_size)
        
        self.gru2 = nn.GRU(hidden_size, hidden_size//2, batch_first=True, num_layers=1)
        self.dropout2 = nn.Dropout(dropout)
        self.bn2 = nn.BatchNorm1d(hidden_size//2)
        
        self.fc1 = nn.Linear(hidden_size//2, 32)
        self.relu = nn.ReLU()
        self.dropout3 = nn.Dropout(dropout)
        
        self.fc2 = nn.Linear(32, 1)
        self.sigmoid = nn.Sigmoid()
        
    def forward(self, x):
        gru_out, _ = self.gru1(x)
        gru_out = self.dropout1(gru_out)
        gru_out = gru_out.permute(0, 2, 1)
        gru_out = self.bn1(gru_out)
        gru_out = gru_out.permute(0, 2, 1)
        
        gru_out, _ = self.gru2(gru_out)
        gru_out = self.dropout2(gru_out[:, -1, :])
        gru_out = self.bn2(gru_out)
        
        out = self.fc1(gru_out)
        out = self.relu(out)
        out = self.dropout3(out)
        
        out = self.fc2(out)
        out = self.sigmoid(out)
        return out

# Pre-calculate Validation Prices for Fast Simulation
val_start_idx = len(X_train) + sequence_length
val_prices = btc_features["Close"].iloc[val_start_idx : val_start_idx + len(y_val_seq)].values

def calculate_return(probs, prices, threshold):
    initial_capital = 10000
    cash = initial_capital
    btc = 0
    fee = 0.001
    
    for i, (prob, price) in enumerate(zip(probs, prices)):
        # Sell everything on last day
        if i == len(probs) - 1:
            cash += btc * price * (1 - fee)
            btc = 0
            continue
        
        val = cash + btc * price
        
        # Target Position Calculation
        target_ratio = 0
        if prob > threshold:
            target_ratio = (prob - 0.5) * 2
            target_ratio = min(target_ratio, 1.0)
            target_ratio = max(0.0, target_ratio)
        
        target_val = val * target_ratio
        current_btc_val = btc * price
        
        # Rebalance
        if target_val > current_btc_val:
            buy_amt = (target_val - current_btc_val) / price
            cost = buy_amt * price
            if cash >= cost * (1+fee):
                btc += buy_amt
                cash -= cost * (1+fee)
        elif target_val < current_btc_val:
            sell_amt = (current_btc_val - target_val) / price
            revenue = sell_amt * price
            btc -= sell_amt
            cash += revenue * (1-fee)
            
    return (cash - initial_capital) / initial_capital * 100

In [ ]:
def objective(trial):
    # Hyperparameters to tune
    hidden_size = trial.suggest_categorical('hidden_size', [32, 64])
    dropout = trial.suggest_float('dropout', 0.1, 0.5)
    lr = trial.suggest_float('lr', 1e-4, 1e-2, log=True)
    threshold = trial.suggest_float('threshold', 0.4, 0.7)
    
    model = ImprovedGRUModel(input_size=X_train_seq.shape[2], hidden_size=hidden_size, dropout=dropout).to(device)
    criterion = nn.BCELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    epochs = 15  # Fast epochs for tuning
    for epoch in range(epochs):
        model.train()
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            out = model(X_batch)
            loss = criterion(out, y_batch.unsqueeze(1))
            loss.backward()
            optimizer.step()
            
    # Evaluate on Validation Set (Profit calc)
    model.eval()
    probs = []
    with torch.no_grad():
        for X_batch, _ in val_loader:
            X_batch = X_batch.to(device)
            out = model(X_batch)
            probs.append(out.cpu().numpy())
    probs = np.vstack(probs).flatten()
    
    profit = calculate_return(probs, val_prices, threshold)
    return profit

print("Starting Optimization (Maximize Return)...")
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=20)

print("Best params:", study.best_params)

# Visualize Hyperparameter Importance
try:
    print("Hyperparameter Importance:")
    fig = plot_param_importances(study)
    fig.show()
except Exception as e:
    print(f"Could not plot importance: {e}")

In [ ]:
# Train Final Model with Best Params
best_params = study.best_params
final_model = ImprovedGRUModel(
    input_size=X_train_seq.shape[2], 
    hidden_size=best_params['hidden_size'], 
    dropout=best_params['dropout']
).to(device)

def train_model(model, train_loader, val_loader, epochs=50, lr=0.001):
    criterion = nn.BCELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    best_val_loss = float('inf')
    patience = 10
    counter = 0
    best_state = None
    
    history = {'train_loss': [], 'val_loss': []}
    
    for epoch in range(epochs):
        model.train()
        train_loss = 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            out = model(X_batch)
            loss = criterion(out, y_batch.unsqueeze(1))
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                out = model(X_batch)
                loss = criterion(out, y_batch.unsqueeze(1))
                val_loss += loss.item()
        
        train_loss /= len(train_loader)
        val_loss /= len(val_loader)
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        
        if (epoch+1) % 5 == 0:
            print(f"Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = model.state_dict()
            counter = 0
        else:
            counter += 1
            if counter >= patience:
                print("Early Stopping")
                break
                
    model.load_state_dict(best_state)
    return history

history = train_model(final_model, train_loader, val_loader, lr=best_params['lr'])

In [ ]:
# Plot Training History
plt.figure(figsize=(10, 6))
plt.plot(history['train_loss'], label='Train Loss')
plt.plot(history['val_loss'], label='Validation Loss')
plt.title('Model Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()

## 4. Evaluation & Visualizations
Confusion Matrix and Performance Metrics with **Best Threshold**

In [ ]:
from utils import plot_confusion_matrix, evaluate_model

# Predictions
def predict_probs(model, loader):
    model.eval()
    probs = []
    with torch.no_grad():
        for X_batch, _ in loader:
            X_batch = X_batch.to(device)
            out = model(X_batch)
            probs.append(out.cpu().numpy())
    return np.vstack(probs).flatten()

probs = predict_probs(final_model, test_loader)

# Use Optimized Threshold
best_threshold = best_params['threshold']
print(f"Using Optimized Threshold: {best_threshold:.4f}")

binary_preds = (probs > best_threshold).astype(int)

# Evaluate
evaluate_model(y_test.values, binary_preds, model_name=f"Optuna GRU (Th={best_threshold:.2f})")
plot_confusion_matrix(y_test.values, binary_preds, model_name=f"Optuna GRU (Th={best_threshold:.2f})")

## 5. Smart Confidence Trading Strategy
Using probability-based position scaling with the optimized model and threshold.

In [ ]:
# Align Dates and Prices for Test Set
test_start_idx = len(btc_features) - len(y_test) + sequence_length
test_prices = btc_features["Close"].iloc[test_start_idx:test_start_idx+len(probs)].values
test_dates = btc_features.index[test_start_idx:test_start_idx+len(probs)]

# Strategy Implementation with Best Threshold
def smart_strategy(probs, prices, dates, threshold, initial_capital=10000, fee=0.001):
    cash = initial_capital
    btc = 0
    portfolio = []
    
    for i, (prob, price) in enumerate(zip(probs, prices)):
        val = cash + btc * price
        portfolio.append(val)
        
        # Sell everything on last day
        if i == len(probs) - 1:
            cash += btc * price * (1 - fee)
            btc = 0
            continue
        
        # Target Position Calculation
        target_ratio = 0
        if prob > threshold:
            target_ratio = (prob - 0.5) * 2
            target_ratio = min(target_ratio, 1.0)
            target_ratio = max(0.0, target_ratio)
        
        target_val = val * target_ratio
        current_btc_val = btc * price
        
        # Rebalance
        if target_val > current_btc_val:
            buy_amt = (target_val - current_btc_val) / price
            cost = buy_amt * price
            if cash >= cost * (1+fee):
                btc += buy_amt
                cash -= cost * (1+fee)
        elif target_val < current_btc_val:
            sell_amt = (current_btc_val - target_val) / price
            revenue = sell_amt * price
            btc -= sell_amt
            cash += revenue * (1-fee)
            
    return portfolio, cash

portfolio_vals, final_cash = smart_strategy(probs, test_prices, test_dates, threshold=best_threshold)
params_return = (final_cash - 10000) / 10000 * 100

# Benchmarks
bh_return = (test_prices[-1] - test_prices[0]) / test_prices[0] * 100

print(f"Smart Strategy Return (Th={best_threshold:.2f}): {params_return:.2f}%")
print(f"Buy & Hold Return: {bh_return:.2f}%")

# Plot
plt.figure(figsize=(12, 6))
plt.plot(test_dates, portfolio_vals, label=f'Smart Strategy (Th={best_threshold:.2f})')
plt.plot(test_dates, [10000 * (p/test_prices[0]) for p in test_prices], label='Buy & Hold', linestyle='--')
plt.legend()
plt.title("Strategy Performance")
plt.show()